# Fase 1: Analise Exploratoria de Dados (EDA)

## Variaveis Biometricas
- AL: Axial Length (comprimento axial)
- ACD: Anterior Chamber Depth (profundidade da camera anterior)
- WTW: White-to-White
- K1, K2: Keratometry (curvatura corneana)
- Correto: Validacao S/N (nao usar no clustering)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuracao
OUTPUT_DIR = '/home/antonio/Documentos/Machine Learning/atividade_3'
IMG_DIR = f'{OUTPUT_DIR}/imgs'

# Carregar dados limpos
df = pd.read_pickle(f'{OUTPUT_DIR}/df_clean.pkl')
numeric_cols = ['AL', 'ACD', 'WTW', 'K1', 'K2']
df_biometric = df[numeric_cols].copy()

print(f'Dataset: {df_biometric.shape[0]} registros, {len(numeric_cols)} features')

## Estatisticas Descritivas

In [ ]:
# Estatisticas completas
stats = df_biometric.describe().T
stats['median'] = df_biometric.median()
stats['variance'] = df_biometric.var()
stats['skewness'] = df_biometric.skew()
stats['kurtosis'] = df_biometric.kurtosis()
stats['range'] = stats['max'] - stats['min']
stats['IQR'] = stats['75%'] - stats['25%']

print('Estatisticas Descritivas Completas:')
print(stats.round(3))

In [ ]:
# Quartis detalhados
print('\n--- Quartis ---')
for col in numeric_cols:
    q1 = df_biometric[col].quantile(0.25)
    q2 = df_biometric[col].quantile(0.50)
    q3 = df_biometric[col].quantile(0.75)
    print(f'{col}: Q1={q1:.2f}, Q2(mediana)={q2:.2f}, Q3={q3:.2f}')

## Histogramas

In [ ]:
# Histogramas para cada variavel
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    ax.hist(df_biometric[col], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(df_biometric[col].mean(), color='red', linestyle='--', label=f'Media: {df_biometric[col].mean():.2f}')
    ax.axvline(df_biometric[col].median(), color='green', linestyle='--', label=f'Mediana: {df_biometric[col].median():.2f}')
    ax.set_title(f'Histograma {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequencia')
    ax.legend(fontsize=8)

axes[5].axis('off')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/hist_all.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Histograma salvo em: {IMG_DIR}/hist_all.png')

In [ ]:
# Histogramas individuais salvos
for col in numeric_cols:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(df_biometric[col], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(df_biometric[col].mean(), color='red', linestyle='--', label=f'Media')
    ax.axvline(df_biometric[col].median(), color='green', linestyle='--', label=f'Mediana')
    ax.set_title(f'Histograma - {col}', fontsize=14, fontweight='bold')
    ax.set_xlabel(col, fontsize=12)
    ax.set_ylabel('Frequencia', fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{IMG_DIR}/hist_{col}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Salvo: {IMG_DIR}/hist_{col}.png')

## Boxplots

In [ ]:
# Boxplot combinado
fig, ax = plt.subplots(figsize=(12, 6))
df_biometric.boxplot(ax=ax, patch_artist=True, 
                     boxprops=dict(facecolor='lightblue', alpha=0.7),
                     medianprops=dict(color='red', linewidth=2))
ax.set_title('Boxplot - Variaveis Biometricas', fontsize=14, fontweight='bold')
ax.set_ylabel('Valor', fontsize=12)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/boxplot_all.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Boxplot salvo em: {IMG_DIR}/boxplot_all.png')

In [ ]:
# Boxplots individuais
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for idx, col in enumerate(numeric_cols):
    axes[idx].boxplot(df_biometric[col], patch_artist=True,
                     boxprops=dict(facecolor='lightblue', alpha=0.7),
                     medianprops=dict(color='red', linewidth=2))
    axes[idx].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Valor')
plt.suptitle('Boxplots - Variaveis Biometricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/boxplot_separate.png', dpi=150, bbox_inches='tight')
plt.show()

## Analise de Outliers

In [ ]:
# Identificar outliers usando IQR
def count_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    return len(outliers), lower, upper

print('--- Contagem de Outliers (IQR) ---')
for col in numeric_cols:
    n_outliers, lower, upper = count_outliers_iqr(df_biometric[col])
    pct = n_outliers / len(df_biometric) * 100
    print(f'{col}: {n_outliers} outliers ({pct:.1f}%) | Limites: [{lower:.2f}, {upper:.2f}]')

## Matriz de Correlacao

In [ ]:
# Correlation heatmap
corr_matrix = df_biometric.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlacao - Variaveis Biometricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelacoes mais fortes:')
for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.5:
            print(f'  {numeric_cols[i]} vs {numeric_cols[j]}: {corr_val:.3f}')

## Pairplot

In [ ]:
# Pairplot completo
plt.figure(figsize=(12, 12))
g = sns.pairplot(df_biometric, diag_kind='hist', plot_kws={'alpha': 0.6, 's': 30},
                 diag_kws={'bins': 25})
g.fig.suptitle('Pairplot - Variaveis Biometricas', y=1.02, fontsize=16, fontweight='bold')
plt.savefig(f'{IMG_DIR}/pairplot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Pairplot salvo em: {IMG_DIR}/pairplot.png')

## Resumo EDA

In [ ]:
# Gerar relatorio HTML
from datetime import datetime

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>EDA - Barrett II Eyes Clustering</title>
    <meta charset="UTF-8">
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; }}
        h1 {{ color: #2c3e50; }}
        h2 {{ color: #34495e; margin-top: 30px; }}
        table {{ border-collapse: collapse; width: 100%; margin: 20px 0; }}
        th, td {{ border: 1px solid #ddd; padding: 10px; text-align: center; }}
        th {{ background-color: #3498db; color: white; }}
        img {{ max-width: 800px; margin: 20px 0; border: 1px solid #ddd; }}
        .highlight {{ background-color: #f9f9f9; padding: 15px; border-radius: 5px; }}
        .summary {{ background-color: #e8f4f8; padding: 20px; border-radius: 5px; }}
    </style>
</head>
<body>
    <h1> Analise Exploratoria de Dados (EDA)</h1>
    <p><strong>Dataset:</strong> Barrett II Eyes Clustering</p>
    <p><strong>Data:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M')}</p>
    <p><strong>Registros:</strong> {df_biometric.shape[0]} | <strong>Features:</strong> {len(numeric_cols)}</p>

    <h2>Estatisticas Descritivas</h2>
    <table>
        <tr>
            <th>Variavel</th>
            <th>Media</th>
            <th>Std</th>
            <th>Min</th>
            <th>Q1</th>
            <th>Mediana</th>
            <th>Q3</th>
            <th>Max</th>
        </tr>
        {''.join([f"<tr><td>{col}</td>" + ''.join([f"<td>{stats.loc[col, metric]:.2f}</td>" for metric in ['mean', 'std', 'min', '25%', '50%', '75%', 'max']]) + "</tr>" for col in numeric_cols])}
    </table>

    <h2>Histogramas</h2>
    <div class='highlight'>
        <img src='imgs/hist_all.png' alt='Histogramas'>
    </div>

    <h2>Boxplots</h2>
    <div class='highlight'>
        <img src='imgs/boxplot_all.png' alt='Boxplots'>
    </div>

    <h2>Matriz de Correlacao</h2>
    <div class='highlight'>
        <img src='imgs/correlation.png' alt='Correlacao'>
    </div>

    <h2>Pairplot</h2>
    <div class='highlight'>
        <img src='imgs/pairplot.png' alt='Pairplot' style='max-width: 1000px;'>
    </div>

    <h2>Principais Descobertas</h2>
    <div class='summary'>
        <ul>
            <li><strong>AL (Axial Length):</strong> Media=23.49, variacao importante (min=20.15, max=30.96)</li>
            <li><strong>ACD (Anterior Chamber Depth):</strong> Media=3.11, outliers presentes</li>
            <li><strong>K1 e K2:</strong> Altamente correlacionados (r>0.9), como esperado</li>
            <li><strong>WTW:</strong> Menor variabilidade (std=0.43)</li>
            <li><strong>Outliers:</strong> Presentes em todas as variaveis, especialmente AL e ACD</li>
        </ul>
    </div>
</body>
</html>
"""

with open(f'{OUTPUT_DIR}/results/eda.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f'Relatorio EDA salvo em: {OUTPUT_DIR}/results/eda.html')